# Exploratory Data Analysis

This notebook joins the menu data with the daily statistics at the correct grain, then explores the merged dataset.

## Join strategy

- `menu_data.email` behaves like a `dou_code` key rather than a personal email.
- `statistics_dataa` is at restaurant level, so several rows can exist for the same `dou_code + date`.
- `menu_data` is effectively one daily menu per `dou_code + date`, repeated across menu items.
- Because of that, the safe join is:
  1. clean and normalize the menu dates,
  2. join the  statistics back to the menu  table.


In [1]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.options.display.float_format = '{:,.2f}'.format
plt.style.use('ggplot')


In [2]:
menu_data = pd.read_excel('../data/raw/Menu_20260411_anonymized.xlsx')
statistics_dataa = pd.read_excel('../data/raw/Statistics_per_day_20260410_anonymized.xlsx')
client_data = pd.read_excel('../data/raw/Statistics_per_day_type_client_20260410_anonymized.xlsx')

print('menu_data:', menu_data.shape)
print('statistics_dataa:', statistics_dataa.shape)
print('client_data:', client_data.shape)


menu_data: (570803, 7)
statistics_dataa: (249837, 9)
client_data: (250249, 21)


In [3]:
menu_data.head()

,meal,quantity,date,meal_type_id,name,email,id
0,كاس حليب,1,2023-11-05 00:00:00,1,ANON_16ed62abf995,11,1
1,مادلين,2,2023-11-05 00:00:00,1,ANON_16ed62abf995,11,1
2,كرواسون,1,2023-11-05 00:00:00,1,ANON_16ed62abf995,11,1
3,فيطاجي,1,2023-11-05 00:00:00,2,ANON_16ed62abf995,11,1
4,عدس,200 غرام,2023-11-05 00:00:00,2,ANON_16ed62abf995,11,1


In [4]:
statistics_dataa.head()

,create_date,resto_name,dou_code,id,id_pro,count,breakfast,launch,dinner
0,2026-04-09,ANON_b520e3d2e6d0,11,7,NaN,276,0,276,0
1,2026-04-09,ANON_b5919475ae88,21,8,"5,185,927.00",131,0,131,0
2,2026-04-09,ANON_8f2b1cc296f2,21,9,"5,185,926.00",407,0,407,0
3,2026-04-09,ANON_79b96b1a5781,31,10,"5,186,007.00",186,0,186,0
4,2026-04-09,ANON_f7895e67904c,41,11,"5,186,084.00",214,0,214,0


In [13]:
menu_data["name"].nunique()

66

name is menu_data is the name of DOU

In [14]:
YEAR_FIX_MAP = {
    '0002': 2023,
    '0004': 2024,
    '0006': 2026,
    '0020': 2024,
    '0023': 2023,
    '0024': 2024,
    '0025': 2025,
    '0026': 2026,
    '0202': 2024,
    '0203': 2024,
    '0204': 2024,
    '0205': 2025,
    '0206': 2026,
    '0224': 2024,
    '0225': 2025,
    '0242': 2024,
}

MEAL_TYPE_LABELS = {
    1: 'breakfast',
    2: 'lunch',
    3: 'dinner',
}


def repair_menu_date(value):
    if pd.isna(value):
        return pd.NaT

    if isinstance(value, pd.Timestamp):
        return value.normalize()

    parsed = pd.to_datetime(value, errors='coerce')
    if pd.notna(parsed) and 2023 <= parsed.year <= 2026:
        return parsed.normalize()

    text = str(value).strip()
    match = re.fullmatch(r'(\d{4})-(\d{2})-(\d{2})', text)
    if not match:
        return pd.NaT

    year_token, month, day = match.groups()
    fixed_year = YEAR_FIX_MAP.get(year_token)

    if fixed_year is None:
        embedded_years = [yy for yy in ('23', '24', '25', '26') if yy in year_token]
        if len(embedded_years) == 1:
            fixed_year = int(f'20{embedded_years[0]}')

    if fixed_year is None:
        return pd.NaT

    return pd.Timestamp(f'{fixed_year}-{month}-{day}')


menu_clean = (
    menu_data.assign(
        dou_code=pd.to_numeric(menu_data['email'], errors='coerce').astype('Int64'),
        date=menu_data['date'].apply(repair_menu_date),
        meal=menu_data['meal'].astype(str).str.strip(),
        meal_type_id=pd.to_numeric(menu_data['meal_type_id'], errors='coerce').astype('Int64'),
    )
    .dropna(subset=['dou_code', 'date'])
)

statistics_clean = statistics_dataa.assign(
    dou_code=pd.to_numeric(statistics_dataa['dou_code'], errors='coerce').astype('Int64'),
    date=pd.to_datetime(statistics_dataa['create_date'], errors='coerce').dt.normalize(),
)

client_clean = client_data.assign(
    dou_code=pd.to_numeric(client_data['dou_code'], errors='coerce').astype('Int64'),
    date=pd.to_datetime(client_data['create_date'], errors='coerce').dt.normalize(),
)

print('Unresolved menu dates after repair:', menu_clean['date'].isna().sum())
display(menu_clean.head())


Unresolved menu dates after repair: 0


,meal,quantity,date,meal_type_id,name,email,id,dou_code
0,كاس حليب,1,2023-11-05,1,ANON_16ed62abf995,11,1,11
1,مادلين,2,2023-11-05,1,ANON_16ed62abf995,11,1,11
2,كرواسون,1,2023-11-05,1,ANON_16ed62abf995,11,1,11
3,فيطاجي,1,2023-11-05,2,ANON_16ed62abf995,11,1,11
4,عدس,200 غرام,2023-11-05,2,ANON_16ed62abf995,11,1,11


## Build Merged dataframe


In [20]:
joined_df = (
    menu_clean.merge(statistics_clean, on=['dou_code', 'date'], how='inner')
    .sort_values(['date', 'dou_code'])
    .reset_index(drop=True)
)

reachable_canteens = (
    statistics_clean.merge(joined_df[['dou_code', 'date']].drop_duplicates(), on=['dou_code', 'date'], how='inner')
    ['resto_name']
    .nunique()
)

coverage_summary = pd.Series({
    'menu_dou_days': len(menu_clean),
    'statistics_dou_days': len(statistics_clean),
    'joined_dou_days': len(joined_df),
    'menu_join_rate_pct': len(joined_df) / len(menu_clean) * 100,
    'statistics_join_rate_pct': len(joined_df) / len(statistics_clean) * 100,
    'linked_dous': joined_df['dou_code'].nunique(),
    'shared_dates': joined_df['date'].nunique(),
    'reachable_canteens': reachable_canteens,
    'stats_only_dous': ', '.join(map(str, sorted(set(statistics_clean['dou_code']) - set(menu_clean['dou_code'])))),
})

display(coverage_summary.to_frame('value'))
display(joined_df.head(n=5))


,value
menu_dou_days,570749
statistics_dou_days,249837
joined_dou_days,3392879
menu_join_rate_pct,594.46
statistics_join_rate_pct,"1,358.04"
linked_dous,66
shared_dates,734
reachable_canteens,506
stats_only_dous,1601


,meal,quantity,date,meal_type_id,name,email,id_x,dou_code,create_date,resto_name,id_y,id_pro,count,breakfast,launch,dinner
0,حليب بالقهوة,150مل,2023-12-12,1,ANON_47514afc3ea2,41,4,41,2023-12-12,ANON_beae590631b8,13,"5,186,082.00",6,0,6,0
1,مادلان,1,2023-12-12,1,ANON_47514afc3ea2,41,4,41,2023-12-12,ANON_beae590631b8,13,"5,186,082.00",6,0,6,0
2,خبز,1,2023-12-12,2,ANON_47514afc3ea2,41,4,41,2023-12-12,ANON_beae590631b8,13,"5,186,082.00",6,0,6,0
3,لوبيا,150غ,2023-12-12,2,ANON_47514afc3ea2,41,4,41,2023-12-12,ANON_beae590631b8,13,"5,186,082.00",6,0,6,0
4,جبن,2,2023-12-12,2,ANON_47514afc3ea2,41,4,41,2023-12-12,ANON_beae590631b8,13,"5,186,082.00",6,0,6,0


NOTE : group different meal items in one row (same date and meal type and resto name should have a column named meal_items that contains all the items),drop quantity, email 

Save joined_df to interim for further analysis

In [ ]:
joined_df.to_excel('../data/interim/joined_canteen_data.xlsx', index=False)